# Experiment 10: Epoch-Based Training with GPU

**Parameters:**
- Buffer Size: 0
- Learning Rate: 0.001
- Min Support: 60%
- Early Stopping: 3 epochs patience

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}")

In [ ]:
# Install dependencies if needed (uncomment if running on Colab)
# pip install tqdm install torch numpy pandas matplotlib tqdm

In [ ]:
# Force CUDA device
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import random
import pickle
from tqdm import tqdm

# Set device to CUDA
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Configuration
CONFIG = {
    "buffer_size": 0,
    "learning_rate": 0.001,
    "min_support_ratio": 0.60,
    "max_epochs": 20,
    "patience": 3,
    "train_episodes": 10000,
    "val_episodes": None,  # Use all
}
print("Configuration:", CONFIG)

In [ ]:
# Import from local modules
from oskp_rl_up_buffer_experiments import (
    BoxPilingEnv, DQNAgent, compute_gt_mask, 
    calculate_average_flat_area, proxy_scores_for_heuristics
)
from experiments import load_instances, evaluate

print("Modules loaded successfully")

In [ ]:
# Load Data
train_data = load_instances("approachesO3DKP/ga_mixed.pt")
val_cut1 = load_instances("approachesO3DKP/cut_1.pt")
val_cut2 = load_instances("approachesO3DKP/cut_2.pt")
val_rs = load_instances("approachesO3DKP/rs.pt")

print(f"Train: {len(train_data)}, CUT1: {len(val_cut1)}, CUT2: {len(val_cut2)}, RS: {len(val_rs)}")

In [ ]:
# Prepare training subset
train_episodes = CONFIG["train_episodes"]
train_subset = train_data[:train_episodes] if train_episodes < len(train_data) else train_data
train_subset = list(train_subset)
print(f"Training on {len(train_subset)} episodes per epoch")

In [ ]:
# Initialize Agent with GPU
env = BoxPilingEnv()
agent = DQNAgent(
    state_dims={"height_map": env.pallet_size, "box_dims": 3},
    action_size=5,
    max_height=env.max_height,
    learning_rate=CONFIG["learning_rate"],
    min_support_ratio=CONFIG["min_support_ratio"],
    require_opposite_edge_support=True,
)

# Force model to CUDA
agent.device = DEVICE
agent.model = agent.model.to(DEVICE)
agent.target_model = agent.target_model.to(DEVICE)
print(f"Agent initialized on {agent.device}")

In [ ]:
# Training function for one epoch
def train_one_epoch_gpu(agent, episodes_boxes, max_buffer_size=0):
    env = BoxPilingEnv()
    total_utilization = 0.0
    heuristic_map = {0: "stacking", 1: "best_fit", 2: "semi_perfect_fit", 3: "random_fit", 4: "corner"}
    
    pbar = tqdm(range(len(episodes_boxes)), desc="Training", unit="ep")
    for episode in pbar:
        state = env.reset()
        done = False
        boxes = episodes_boxes[episode]
        box_idx = 0
        buffer = []
        
        while not done and box_idx < len(boxes):
            box_dims = boxes[box_idx]
            box_idx += 1
            state = env.new_box_arrival(box_dims)
            
            pred_mask = agent.predict_mask(state, buffer_count=len(buffer))
            mask_bias = proxy_scores_for_heuristics(env, pred_mask)
            h_idx = agent.act_with_mask_bias(state, buffer_count=len(buffer), mask_bias=mask_bias, beta=0.5)
            heuristic = heuristic_map[h_idx]
            
            action, _ = env.choose_action_by_heuristic(heuristic, pred_mask=pred_mask)
            if action is None:
                action, _ = env.choose_action_by_heuristic(heuristic, pred_mask=None)
                if action is None:
                    agent.remember(state, h_idx, -5.0, state, False, len(buffer))
                    agent.replay()
                    if len(buffer) < max_buffer_size:
                        buffer.append(box_dims)
                    else:
                        done = True
                        break
                    continue
            
            next_state, reward, local_done, _ = env.step(action)
            agent.remember(state, h_idx, reward, next_state, local_done, len(buffer))
            agent.replay()
            state = next_state
            
            if env._is_terminal():
                done = True
                break
        
        # Metrics
        utilization = env.current_height_map.sum() / (env.pallet_size[0] * env.pallet_size[1] * env.max_height)
        total_utilization += utilization
        
        # Update epsilon
        if agent.epsilon > agent.epsilon_min:
            agent.epsilon *= agent.epsilon_decay
        
        # Update target network
        if episode % 10 == 0:
            agent.update_target_model()
        
        pbar.set_postfix({"Util": f"{utilization:.1%}", "Eps": f"{agent.epsilon:.3f}"})
    
    return total_utilization / len(episodes_boxes)

In [ ]:
# Epoch-based training loop
epoch_results = []
best_avg_util = 0.0
epochs_without_improvement = 0
env_params = {"max_buffer_size": CONFIG["buffer_size"], "min_support_ratio": CONFIG["min_support_ratio"]}

for epoch in range(1, CONFIG["max_epochs"] + 1):
    print(f"\n{"="*50}")
    print(f"EPOCH {epoch}/{CONFIG["max_epochs"]}")
    print(f"{"="*50}")
    
    # Shuffle training data
    random.shuffle(train_subset)
    
    # Train one epoch
    avg_train_util = train_one_epoch_gpu(agent, train_subset, CONFIG["buffer_size"])
    print(f"\nEpoch Training Avg Utilization: {avg_train_util:.2%}")
    
    # Validate
    print("Validating...")
    s_cut1 = evaluate(agent, val_cut1, env_params)
    s_cut2 = evaluate(agent, val_cut2, env_params)
    s_rs = evaluate(agent, val_rs, env_params)
    avg_util = (s_cut1 + s_cut2 + s_rs) / 3
    
    print(f"  CUT-1: {s_cut1:.2%}, CUT-2: {s_cut2:.2%}, RS: {s_rs:.2%}")
    print(f"  Average Validation Utilization: {avg_util:.2%}")
    
    epoch_results.append({
        "epoch": epoch,
        "train_util": avg_train_util,
        "cut1": s_cut1,
        "cut2": s_cut2,
        "rs": s_rs,
        "avg_util": avg_util,
        "epsilon": agent.epsilon
    })
    
    # Early stopping check
    if avg_util > best_avg_util + 0.001:
        best_avg_util = avg_util
        epochs_without_improvement = 0
        agent.save_model("best_model_gpu.pt")
        print(f"  ★ New best model! (Avg: {best_avg_util:.2%})")
    else:
        epochs_without_improvement += 1
        print(f"  No improvement ({epochs_without_improvement}/{CONFIG["patience"]})")
    
    if epochs_without_improvement >= CONFIG["patience"]:
        print(f"\n=== EARLY STOPPING ===")
        break

print(f"\n=== TRAINING COMPLETE ===")
print(f"Best Avg Utilization: {best_avg_util:.2%}")

In [ ]:
# Results summary
results_df = pd.DataFrame(epoch_results)
print(results_df)
results_df.to_csv("epoch_training_results_gpu.csv", index=False)

In [ ]:
# Plot learning curve
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(results_df["epoch"], results_df["avg_util"], marker="o")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Avg Validation Utilization")
ax[0].set_title("Learning Curve")
ax[0].grid(True)

ax[1].plot(results_df["epoch"], results_df["epsilon"], marker="o", color="orange")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Epsilon")
ax[1].set_title("Exploration Decay")
ax[1].grid(True)

plt.tight_layout()
plt.savefig("learning_curve_gpu.png")
plt.show()